# Document-level fact Knowledge Indicators + an agent harness

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kderusso/elasticsearch-labs/blob/kderusso/context-engine-notebooks/notebooks/context-engine/manual-walkthrough/index-facts-kis.ipynb)

The **Context Engine** gives AI agents structured, pre-computed **Knowledge Indicators (KIs)** — facts they can retrieve in a single call instead of reading whole documents — stored on Elastic's hybrid search stack and usable from any agent framework. This notebook creates one KI **per document** so an agent can *retrieve a fact*, then closes the full loop the Context Engine is built for:

> **data in → one fact KI per document (via a workflow) → query via `get_context` → consumed as a tool by an external agent**

KIs come in two flavors: **document-level** KIs like these capture individual facts an agent can retrieve, while **index-level** KIs profile a whole index for routing. This notebook covers the document-level flow end to end.

We use the [BrowseComp-Plus](https://github.com/texttron/BrowseComp-Plus) corpus, indexed BM25-only for speed.

### What you'll do

1. Connect to a local Elasticsearch + Kibana and **explicitly enable the Context Engine feature flag**.
2. Index a small slice of the BrowseComp-Plus corpus (BM25 only).
3. Run a **Kibana Workflow** that, for each document, chains `ai.agent` (structured-output KI extraction) → `contextEngine.addEntry`.
4. Answer a question by doing a plain **BM25 search** against the raw index — the baseline without the Context Engine.
5. Run a **Kibana Workflow** to create one fact KI per document.
6. Answer the **same question** through the Context Engine and compare: structured KIs vs raw document chunks.
7. Wrap `get_context` as a **tool** inside a [LangChain deep agents](https://pypi.org/project/deepagents/) harness and watch the agent answer a question by retrieving from the Context Engine.
6. Clean up.

> ℹ️ **Prerequisites.** An **Elastic Cloud** deployment where the Context Engine is available, a **GenAI connector** configured in Kibana for the `ai.agent` step, and an **OpenRouter API key** for the agent harness (step 6).


## 1. Set up the environment

Install the Python dependencies. `langchain-openai` + `deepagents` power the agent harness in step 6.

In [ ]:
!pip install -q "elasticsearch>=9,<10" datasets requests pyyaml langchain-openai langgraph deepagents

### Connect to Elasticsearch and Kibana

The Context Engine, Workflows, and the feature flag live in **Kibana**; the dataset is indexed into **Elasticsearch**. We connect to both with your **Elastic Cloud ID** and an **API key** — the Kibana URL is derived from the Cloud ID, so no other connection details are needed.

- [Find your Cloud ID](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id)
- [Create an API key](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key)

In [ ]:
import json
import time
import base64
import requests
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic API key: ")

# Elasticsearch client — used to index the dataset.
client = Elasticsearch(cloud_id=ELASTIC_CLOUD_ID, api_key=ELASTIC_API_KEY)
print(client.info())

We derive the Kibana URL from the Cloud ID and authenticate with the API key (`Authorization: ApiKey …`). This helper also sets `kbn-xsrf` for mutating requests, `x-elastic-internal-origin` for **internal** routes (the Context Engine search API), and `elastic-api-version` for the **public, versioned** Workflows API.

In [ ]:
# Kibana shares the deployment's hostname with Elasticsearch; derive its URL from
# the Cloud ID, which decodes to "<host>$<es-uuid>$<kibana-uuid>".
def _kibana_url_from_cloud_id(cloud_id):
    _, encoded = cloud_id.split(":", 1)
    host, _es_uuid, kibana_uuid = base64.b64decode(encoded).decode().split("$")
    return f"https://{kibana_uuid}.{host}"

KIBANA_URL = _kibana_url_from_cloud_id(ELASTIC_CLOUD_ID)

def kbn_request(method, path, *, body=None, internal=False, api_version=None):
    """Call a Kibana REST API and return the parsed JSON response."""
    headers = {
        "Authorization": f"ApiKey {ELASTIC_API_KEY}",
        "kbn-xsrf": "true",
        "Content-Type": "application/json",
    }
    if internal:
        headers["x-elastic-internal-origin"] = "Kibana"
    if api_version:
        headers["elastic-api-version"] = api_version
    resp = requests.request(
        method,
        f"{KIBANA_URL}{path}",
        headers=headers,
        data=json.dumps(body) if body is not None else None,
    )
    resp.raise_for_status()
    return resp.json() if resp.text else {}

status = kbn_request("GET", "/api/status")
print("Kibana status:", status.get("status", {}).get("overall", {}).get("level"))

### Enable the Context Engine feature flag

The Context Engine is gated behind the `agentBuilder:experimentalFeatures` advanced setting. **Don't assume it's on** — set it explicitly.

> If the cell prints "Could not set feature flag", the setting may already be on or is managed at the deployment level. Check **Stack Management → Advanced Settings** and search for `agentBuilder:experimentalFeatures` to confirm it's enabled before continuing.

In [ ]:
FEATURE_FLAGS = {
    "agentBuilder:experimentalFeatures": True,
    "agentContextLayer:experimentalFeatures": True,
}

try:
    kbn_request("POST", "/internal/kibana/settings", body={"changes": FEATURE_FLAGS}, internal=True)
    print("Enabled feature flags:", list(FEATURE_FLAGS))
except Exception as e:
    print(f"Could not set feature flags via API: {e}")
    print("This is expected if the settings are already enabled or managed at the deployment level.")
    print("Verify they are ON in Stack Management → Advanced Settings before continuing.")

### OpenRouter API key

We use an OpenRouter key to call a language model — both for the before/after comparison and for the agent harness later.

In [ ]:
from langchain_openai import ChatOpenAI

OPENROUTER_API_KEY = getpass("OpenRouter API key: ")
OPENROUTER_MODEL = "anthropic/claude-haiku-4-5"

llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    max_tokens=512,
)

def answer_question(question, context_chunks):
    """Synthesize an answer from a list of context strings."""
    ctx = "\n\n---\n\n".join(context_chunks)
    prompt = (
        "Answer the question using only the context provided. "
        "Be concise. If the context does not contain the answer, say so.\n\n"
        f"Context:\n{ctx}\n\nQuestion: {question}"
    )
    return llm.invoke(prompt).content

## 2. Index a small BrowseComp-Plus dataset (BM25 only)

The [BrowseComp-Plus corpus](https://huggingface.co/datasets/Tevatron/browsecomp-plus-corpus) is ~100k web documents (~1.76 GB); each row has `docid`, `text`, and `url`. We **stream** the dataset and take the first `SAMPLE_DOCS` rows into a **BM25-only** index — plain `text` plus `keyword` metadata, no vectors. The workflow's ES|QL query reads a `title` field, so we derive a lightweight title from the first line of each document and attach mapping metadata so the index is self-describing.

The cell is **idempotent**: if the index already exists with documents, it's reused and indexing is skipped — so you can reuse the same index without re-running setup.

In [ ]:
INDEX_NAME = "browsecomp-plus"
SAMPLE_DOCS = 50  # documents to index (we generate one KI per indexed doc)

if client.indices.exists(index=INDEX_NAME) and client.count(index=INDEX_NAME)["count"] > 0:
    count = client.count(index=INDEX_NAME)["count"]
    print(f"Index '{INDEX_NAME}' already has {count} documents — reusing it (skipping indexing).")
else:
    # BM25-only mapping, enriched with descriptions; `title` is derived from the body.
    client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
    client.indices.create(
        index=INDEX_NAME,
        mappings={
            "_meta": {
                "description": (
                    "BrowseComp-Plus corpus: ~100k human-verified web documents "
                    "(news articles, Wikipedia entries, institutional pages) used as a "
                    "reasoning-intensive browsing/QA retrieval benchmark. BM25-only index."
                )
            },
            "properties": {
                "docid": {"type": "keyword", "meta": {"description": "Stable corpus document id."}},
                "url": {"type": "keyword", "meta": {"description": "Source URL the document was crawled from."}},
                "title": {"type": "text", "meta": {"description": "Document title (first line of the body)."}},
                "text": {"type": "text", "meta": {"description": "Full document text: title, date, and body content."}},
            },
        },
    )

    from datasets import load_dataset

    # Stream so we only pull the first SAMPLE_DOCS rows instead of the full ~1.76 GB corpus.
    corpus = load_dataset("Tevatron/browsecomp-plus-corpus", split="train", streaming=True)

    def actions(n):
        for i, row in enumerate(corpus):
            if i >= n:
                break
            text = row["text"]
            title = text.strip().split("\n", 1)[0][:120]  # first line as a stand-in title
            yield {
                "_index": INDEX_NAME,
                "_id": row["docid"],
                "_source": {"docid": row["docid"], "url": row["url"], "title": title, "text": text},
            }

    helpers.bulk(client, actions(SAMPLE_DOCS))
    client.indices.refresh(index=INDEX_NAME)
    print(f"Indexed {client.count(index=INDEX_NAME)['count']} documents into '{INDEX_NAME}'.")

## 3. Answer the question by searching the index directly

Before creating any KIs, we answer the same question using a plain BM25 search against the raw index. The top document chunks are passed directly to an LLM.

In [ ]:
QUESTION = "What was the actress who played Torvi from Vikings also known for?"

resp = client.search(
    index=INDEX_NAME,
    body={
        "query": {"multi_match": {"query": QUESTION, "fields": ["title", "text"]}},
        "size": 5,
        "_source": ["title", "text"],
    },
)

chunks = []
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    chunks.append(src.get('title', '') + "\n" + src.get('text', '')[:600])

for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()
print("=== Answer from BM25 search (raw document chunks) ===")
print()
print(answer_question(QUESTION, chunks))

## 4. Create one fact KI per document with a Workflow

This workflow looks up a **single document by `docid`** (passed in as an input) with ES|QL, then `foreach` (a one-row loop) runs two steps:

| Step | Type | What it does |
|------|------|--------------|
| `generate_ki` | `ai.agent` | Read the document and extract a retrieval-optimized Knowledge Indicator as **structured output** (title, summary, questions it answers, key entities, topics, tagline). |
| `sink_ki` | `contextEngine.addEntry` | Upsert the generated KI into the Context Engine, namespaced under the `corpus_entry` type. |

`contextEngine.addEntry` decides **what lands in the engine**. Both `content` and `description` are indexed as `semantic_text`, so the workflow gives them complementary surfaces: `content` carries a provenance header (which index + docid + URL back this KI) plus the dense factual summary and the questions the doc answers; `description` carries a second topical angle (tagline + topics + entities). `originId` is the stable `docid`, so re-runs idempotently replace the prior chunk.

We create the workflow **once**, then run one execution **per document**. Because each execution is self-contained, we fan them out **in parallel** from the client (next section) so the per-doc `ai.agent` calls run concurrently instead of one at a time.

> This is the demo workflow, adapted minimally: it processes one document per run (the demo looped over all docs in a single run) so executions can be parallelized, and the `elasticsearchIndices` gating field is dropped (your local `contextEngine.addEntry` step exposes only `permissions`).

### Point the workflow at your GenAI connector (optional)

Leave blank to use Kibana's **default** GenAI connector, or paste a specific connector id to pin one.

In [ ]:
LLM_CONNECTOR_ID = input("Kibana GenAI connector id (blank = default connector): ").strip()

### Define the workflow

In [ ]:
_WORKFLOW_YAML_TEMPLATE = """
version: '1'
name: browsecomp-plus-per-doc-ki
description: Look up a single BrowseComp-Plus document by id with ES|QL, generate a KI with an AI agent, and sink it into the Context Engine as a corpus_entry.
enabled: true
tags:
  - context-engine
  - browsecomp-plus
triggers:
  - type: manual
steps:
  - name: query_corpus
    type: elasticsearch.esql.query
    with:
      # One workflow execution handles a single document (passed in as inputs.docid),
      # so the client can fan many executions out in parallel.
      # Column order drives the foreach.item[N] indices below:
      #   item[0]=docid  item[1]=title  item[2]=url  item[3]=text
      # SUBSTRING keeps the prompt bounded.
      query: >
        FROM __INDEX_NAME__
        | WHERE docid == "{{ inputs.docid }}"
        | KEEP docid, title, url, text
        | EVAL text = SUBSTRING(text, 1, 12000)
        | LIMIT 1

  - name: loop_corpus_docs
    type: foreach
    foreach: '{{steps.query_corpus.output.values}}'
    steps:
      # Turn the raw doc into a retrieval-optimized Knowledge Indicator (structured output).
      - name: generate_ki
        type: ai.agent
        connector-id: "__LLM_CONNECTOR_ID__"
        timeout: 120s
        with:
          message: >
            You are a knowledge engineer building a Knowledge Indicator (KI) for an
            enterprise document-retrieval corpus. A KI is a compact, high-signal record
            that a hybrid (BM25 + semantic) search engine and an AI agent use to FIND and
            JUDGE the source document without reading it in full.

            Read the document below and extract a faithful, richly structured KI. Be 100%
            grounded: never state anything not supported by the text. Prefer concrete,
            named specifics (people, organizations, products, dates, places, figures) over
            vague phrasing. If a field cannot be determined from the text, return an empty
            string or empty array rather than guessing.

            Document ID: {{ foreach.item[0] }}
            Original Title: {{ foreach.item[1] }}
            Source URL: {{ foreach.item[2] }}
            Document Body:
            {{ foreach.item[3] }}
          schema:
            type: object
            properties:
              title:
                type: string
                description: A concise, specific, human-readable title (<= 12 words).
              summary:
                type: string
                description: A dense 3-5 sentence factual summary capturing the document's main claims, named entities, and conclusions. The PRIMARY semantic search surface.
              answers_questions:
                type: array
                items:
                  type: string
                description: 2-5 natural-language questions this document can authoritatively answer.
              key_entities:
                type: array
                items:
                  type: string
                description: 3-10 salient named entities (people, organizations, products, places, dates) explicitly mentioned.
              topics:
                type: array
                items:
                  type: string
                description: 3-8 short topic/category labels describing what the document is about.
              tagline:
                type: string
                description: A single ultra-short phrase (<= 6 words) capturing the essence of the document.
            required:
              - title
              - summary
              - answers_questions
              - key_entities
              - topics

      # Sink the generated KI into the Context Engine.
      - name: sink_ki
        type: contextEngine.addEntry
        with:
          originId: '{{ foreach.item[0] }}'
          attachmentType: corpus_entry
          action: upsert
          chunks:
            - type: corpus_entry
              title: '{{ foreach.item[1] | default: steps.generate_ki.output.structured_output.title }}'
              content: >
                === SOURCE / PROVENANCE ===
                Backing Elasticsearch index: __INDEX_NAME__
                Document ID (docid): {{ foreach.item[0] }}
                Source URL: {{ foreach.item[2] }}
                Retrieve the full original document with ES|QL:
                FROM __INDEX_NAME__ | WHERE docid == "{{ foreach.item[0] }}"
                === KNOWLEDGE INDICATOR ===
                {{ steps.generate_ki.output.structured_output.summary }}
                Questions this document answers: {{ steps.generate_ki.output.structured_output.answers_questions | join: " | " }}
                Key entities: {{ steps.generate_ki.output.structured_output.key_entities | join: ", " }}
              description: >
                {{ steps.generate_ki.output.structured_output.tagline }}.
                Topics: {{ steps.generate_ki.output.structured_output.topics | join: ", " }}.
                Entities: {{ steps.generate_ki.output.structured_output.key_entities | join: ", " }}.
"""

# Point the workflow at our index (single source of truth = INDEX_NAME).
_yaml = _WORKFLOW_YAML_TEMPLATE.replace("__INDEX_NAME__", INDEX_NAME)

# Pin a connector if one was supplied; otherwise drop the line so ai.agent uses
# Kibana's default GenAI connector (step-level config can't be templated).
if LLM_CONNECTOR_ID:
    WORKFLOW_YAML = _yaml.replace("__LLM_CONNECTOR_ID__", LLM_CONNECTOR_ID)
else:
    WORKFLOW_YAML = "\n".join(
        line for line in _yaml.splitlines() if "__LLM_CONNECTOR_ID__" not in line
    )

print(WORKFLOW_YAML)

### Create and run the workflow

We create the workflow once, then (in the next cell) run one execution per document, up to `MAX_PARALLEL_KIS` at a time. Each execution makes one `ai.agent` call, so running them in parallel is what keeps the whole pass fast.

In [ ]:
WF_API_VERSION = "2023-10-31"
TERMINAL_STATES = {"completed", "failed", "cancelled", "timed_out", "skipped"}

def create_workflow(yaml_str):
    return kbn_request("POST", "/api/workflows/workflow",
                       body={"yaml": yaml_str}, api_version=WF_API_VERSION)

def run_workflow(workflow_id, inputs):
    return kbn_request("POST", f"/api/workflows/workflow/{workflow_id}/run",
                       body={"inputs": inputs}, api_version=WF_API_VERSION)

def wait_for_execution(execution_id, timeout=600, interval=5):
    deadline = time.time() + timeout
    while time.time() < deadline:
        ex = kbn_request("GET", f"/api/workflows/executions/{execution_id}?includeOutput=true",
                         api_version=WF_API_VERSION)
        if ex["status"] in TERMINAL_STATES:
            return ex
        time.sleep(interval)
    raise TimeoutError(f"Execution {execution_id} did not finish within {timeout}s")

wf = create_workflow(WORKFLOW_YAML)
workflow_id = wf["id"]
print("Created workflow:", workflow_id)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_PARALLEL_KIS = 8   # how many workflow executions to run at once
PROGRESS_EVERY = 10    # print a running count every N completed KIs
MAX_ATTEMPTS = 3       # per-doc attempts before treating a failure as real
RETRY_BACKOFF = 5      # seconds between attempts, multiplied by the attempt number

# One execution per document — fan them out so the LLM calls run concurrently.
docids = [
    hit["_id"]
    for hit in helpers.scan(client, index=INDEX_NAME, _source=False, query={"query": {"match_all": {}}})
]
print(f"Generating one KI per document for {len(docids)} docs ({MAX_PARALLEL_KIS} in parallel)...\n")

def generate_ki_for(docid):
    """Run the workflow for one doc, retrying transient failures up to MAX_ATTEMPTS.

    The ai.agent inference call fails intermittently (e.g. a transient empty-content
    400), so a single failure isn't necessarily a real one — retry before giving up.
    """
    last = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            execution = run_workflow(workflow_id, {"docid": docid})
            result = wait_for_execution(execution["workflowExecutionId"])
            if result["status"] == "completed":
                return docid, result
            last = result
        except Exception as e:
            last = {"status": "error", "error": {"message": str(e)}}
        if attempt < MAX_ATTEMPTS:
            print(f"  retry {attempt}/{MAX_ATTEMPTS - 1} for docid={docid} (was: {last['status']})")
            time.sleep(RETRY_BACKOFF * attempt)
    return docid, last  # retries exhausted — caller treats this as a real failure

completed = 0
failure = None
with ThreadPoolExecutor(max_workers=MAX_PARALLEL_KIS) as pool:
    futures = {pool.submit(generate_ki_for, d): d for d in docids}
    for future in as_completed(futures):
        docid, result = future.result()
        if result["status"] != "completed":
            # Abort only after retries are exhausted: cancel anything not yet started so
            # we stop creating KIs until the problem is fixed. (In-flight runs still finish.)
            failure = (docid, result)
            for f in futures:
                f.cancel()
            break
        completed += 1
        if completed % PROGRESS_EVERY == 0 or completed == len(docids):
            print(f"  {completed}/{len(docids)} KIs completed")

if failure:
    docid, result = failure
    print(f"\nAborted — docid={docid} still failing after {MAX_ATTEMPTS} attempts (status: {result['status']}).")
    if result.get("error"):
        print("Error:", result["error"].get("message", result["error"]))
    for step in result.get("stepExecutions", []):
        if step.get("status") == "failed":
            print("Failed step:", step.get("stepId"), "->", json.dumps(step.get("error")))
    raise RuntimeError(f"Aborted KI generation: docid={docid} failed after {MAX_ATTEMPTS} attempts; fix the issue and re-run.")

print(f"\nDone: {completed}/{len(docids)} KIs completed.")

## 5. Answer the question from KIs

Now we ask the same question through the Context Engine. Each result is a pre-computed Knowledge Indicator: structured, summarized, and semantically indexed by the workflow. We show the KIs returned, then synthesize an answer from their content.

In [ ]:
def get_context(query, size=5, types=None):
    body = {
        "query": query,
        "size": size,
        "fields": ["content", "description", "tags", "references"],
    }
    if types:
        body["filters"] = {"types": types}
    return kbn_request("POST", "/internal/agent_context_layer/sml/_search",
                       body=body, internal=True)

response = get_context(QUESTION, types=["corpus_entry"])
ki_results = response.get("results", [])

print(f"Knowledge Indicators retrieved: {len(ki_results)}\n")
for ki in ki_results:
    print(f"  {ki['title']}")
    print(f"  {ki.get('description', '')[:150]}")
    print()

chunks = [f"{r['title']}\n{r.get('content', '')}" for r in ki_results]
for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()
print("=== Answer from Context Engine KIs ===")
print()
print(answer_question(QUESTION, chunks))

### Fetch a single KI by id via the SML API

The `_search` endpoint ranks KIs; the SML API also lets you fetch one entry **by id**. We take the top KI from the search above and issue `GET /internal/agent_context_layer/sml/{id}` to display its full stored record.

In [ ]:
# Pick one KI (the top search hit) and fetch its full record directly by id via the SML API.
hits = get_context(QUESTION, size=1, types=["corpus_entry"])["results"]
if not hits:
    print("No KIs found — run the workflow cells above first.")
else:
    ki_id = hits[0]["id"]
    print(f"GET /internal/agent_context_layer/sml/{ki_id}\n")
    ki = kbn_request("GET", f"/internal/agent_context_layer/sml/{ki_id}", internal=True)
    print(json.dumps(ki, indent=2))

## 6. Wrap `get_context` as a tool for a deep agent harness

The point of the Context Engine is to be **consumed by an agent**. Here we wrap `get_context` as a LangChain tool, give the agent system-prompt guidance on *when* to call it, and use a **deep agent** ([`deepagents`](https://pypi.org/project/deepagents/)) to answer a question by retrieving from the engine.

Define the tool. The docstring is what the agent reads to decide when to call it — so it states the *when*, not just the *what*.

In [ ]:
from langchain_core.tools import tool

@tool
def search_context(query: str) -> str:
    """Search the Context Engine for Knowledge Indicators (KIs) extracted from the document corpus.

    Call this whenever answering depends on specific facts, names, dates, organizations, or
    events that would live in the corpus rather than in your own training data. Pass a focused
    natural-language query. Returns the most relevant KIs with their title and content
    (each KI includes provenance so you can cite the source document).
    """
    results = get_context(query, size=5, types=["corpus_entry"])["results"]
    if not results:
        return "No relevant context found in the Context Engine."
    return "\n\n".join(f"[{r['title']}]\n{r.get('content', '')}" for r in results)

Create the agent with system-prompt guidance, then ask it a question. The agent decides on its own to call `search_context`, reads the returned KIs, and answers from them.

In [ ]:
from deepagents import create_deep_agent

SYSTEM_PROMPT = """You are a research assistant answering questions about a document corpus.
You have NOT memorized the corpus. Whenever a question depends on specific facts, names, dates,
organizations, or events, call the `search_context` tool to retrieve relevant Knowledge Indicators
before answering. Ground your answer strictly in what the tool returns, and cite the KI titles
you used. If the retrieved context does not contain the answer, say so plainly rather than guessing."""

# The deep agent plans and may chain several tool calls, so it needs more output headroom than
# the shared `llm` (max_tokens=512, tuned for the concise answers in steps 3 & 5). Reuse the same
# OpenRouter model + key with a roomier token budget — this leaves `llm` untouched.
agent_llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    max_tokens=4096,
)

agent = create_deep_agent(
    model=agent_llm,
    tools=[search_context],
    system_prompt=SYSTEM_PROMPT,
)

result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print(result["messages"][-1].content)

That is the full Context Engine loop: documents went in, the workflow distilled each into a fact KI, `get_context` made them retrievable, and an external agent answered a question by calling `get_context` as a tool and grounding its response in the returned KIs.

## 7. Clean up

Remove the KIs from the Context Engine, delete the workflow, and drop the Elasticsearch index.

In [ ]:
# 1) Delete the corpus_entry KIs we created (look them up, then delete by id).
seen = set()
for hit in get_context("knowledge indicator document", size=100, types=["corpus_entry"])["results"]:
    if hit["id"] in seen:
        continue
    seen.add(hit["id"])
    kbn_request("DELETE", f"/internal/agent_context_layer/sml/{hit['id']}", internal=True)
    print("Deleted KI:", hit["id"])

# 2) Delete the workflow.
try:
    kbn_request("DELETE", f"/api/workflows/workflow/{workflow_id}", api_version=WF_API_VERSION)
    print("Deleted workflow:", workflow_id)
except Exception as e:
    print("Workflow delete skipped:", e)

# 3) Delete the Elasticsearch index.
client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
print("Deleted index:", INDEX_NAME)

# The feature flag is left enabled. To turn it back off:
# kbn_request("POST", "/api/kibana/settings", body={"changes": {FEATURE_FLAG: False}})